In [1]:
# LIBRERÍAS
!pip install -q python-docx reportlab matplotlib
!apt-get -qq update
!apt-get -qq install -y libreoffice

import pandas as pd
import os
import matplotlib.pyplot as plt
from docx import Document
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import letter
import locale

try:
    locale.setlocale(locale.LC_ALL, 'en_US.UTF-8')
except:
    locale.setlocale(locale.LC_ALL, '')


# 2. CONFIGURACIÓN GENERAL


BASE_EXCEL = "/content/Plantilla_Reporte_TuCatastro.xlsx"
PLANTILLA_WORD = "/content/Mpio_Informe_tramites_Catastral_ACC.docx"

ANIOS = [2024, 2025]
CARPETA_RAIZ = "/content/Reportes_ACC_2024_2025"


# 3. NORMALIZACIÓN DE COLUMNAS

def normalizar_columnas(df):
    df.columns = df.columns.str.strip()
    df.columns = (
        df.columns
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )
    return df


def estandarizar_estado(df, col_estado="Estado"):
    """
    Renombra valores específicos de la columna Estado antes de cálculos.
    Opera case-insensitive y preserva NaN.
    """
    if col_estado not in df.columns:
        print(f"⚠ No existe la columna '{col_estado}'. No se aplicó estandarización de Estado.")
        return df

    mapeo = {
        "finalizado_por_desistimiento": "Finalizado por desistimiento",
        "generacion_acto_administrativo": "Generacion de acto administrativo",
        "informado_notificado_cumplido": "Informado notificado cumplido",
        "no_procedente": "No procedente",
        "radicado": "Radicado",
    }

    df = df.copy()
    # Normaliza a string para mapear, pero sin romper NaN
    s = df[col_estado].astype(str).str.strip()
    s_low = s.str.lower()

    # Aplica reemplazo (solo donde coincide exactamente con las claves)
    df[col_estado] = s_low.map(mapeo).fillna(df[col_estado])
    return df


def filtrar_tipos_excluidos(df, columna_tipo="Tipo", excluidos=("Certificado", "Carta")):
    """
    Elimina filas donde df[columna_tipo] contenga cualquiera de las palabras en 'excluidos'
    (case-insensitive). Los valores NaN en Tipo se conservan.
    """
    if columna_tipo not in df.columns:
        # Si no existe la columna, no filtramos para no romper el proceso
        print(f"⚠ No existe la columna '{columna_tipo}'. No se aplicó filtro de tipos excluidos.")
        return df

    patron = "|".join([str(x) for x in excluidos])  # "Certificado|Carta"
    mask_excluir = df[columna_tipo].astype(str).str.contains(patron, case=False, na=False)
    return df[~mask_excluir].copy()


def pct(numerador, denominador):
    if denominador in (0, None) or pd.isna(denominador):
        return 0.0
    return (numerador / denominador) * 100.0

def construir_textos_analisis(df_mun, mun, anios=(2024, 2025)):
    # Total radicados (únicos) en el periodo
    total = int(df_mun['Numero radicado'].nunique())

    # Finalizados: asumimos que "Finalizado" y "Informado notificado cumplido"
    # representan trámites con resolución / cierre. Ajusta si tu definición es otra.
    # Usamos Estado estandarizado (ya renombrado).
    estados_finalizados = {"Finalizado", "Informado notificado cumplido", "Finalizado por desistimiento"}
    finalizados = int(df_mun[df_mun['Estado'].eq("Informado notificado cumplido")]['Numero radicado'].nunique())

    # % finalizados sobre total
    pct_finalizados = pct(finalizados, total)

    # Para el segundo párrafo: territorio vs central desde OFICINA DE GESTION
    # Criterio: si contiene "CENTRAL" => Central, si no => Territorio
    territorio = int(df_mun[~df_mun['OFICINA DE GESTION'].astype(str).str.upper().str.contains("CENTRAL", na=False)]
                     ['Numero radicado'].nunique())
    central = int(df_mun[df_mun['OFICINA DE GESTION'].astype(str).str.upper().str.contains("CENTRAL", na=False)]
                  ['Numero radicado'].nunique())

    pct_territorio = pct(territorio, total)

    texto_pre_tabla1 = (
        f"La siguiente tabla corresponde al estado de los radicados por año y con un total general. "
        f"En {mun} se presenta un total de {total} radicados entre los años {anios[0]} y {anios[1]}. "
        f"Los valores se desagregan para los años {anios[0]} y {anios[1]}. "
        f"Para estos dos años se han finalizado el {pct_finalizados:.0f}% de los trámites recibidos."
    )

    texto_post_tabla1 = (
        f"De los {total} radicados del municipio de {mun}, más del {pct_territorio:.1f}% "
        f"representan trámites que el enlace territorial de la ACC resuelve en el municipio "
        f"y el porcentaje restante se resuelve a nivel central."
    )

    return texto_pre_tabla1, texto_post_tabla1

def reemplazar_marcador_preservando_formato(doc, marcador, texto_nuevo):
    """
    Reemplaza un marcador (ej. [[DESC_RADICADOS]]) por texto_nuevo sin destruir estilo.
    Si el marcador está en un run, se reemplaza dentro del run. Si está dividido en varios runs,
    se hace una sustitución robusta a nivel de párrafo (reconstruyendo runs, intentando preservar estilo base).
    """
    for p in doc.paragraphs:
        if marcador in p.text:
            # Caso simple: el marcador está dentro de un run
            for run in p.runs:
                if marcador in run.text:
                    run.text = run.text.replace(marcador, texto_nuevo)
                    return True

            # Caso robusto: marcador dividido en runs
            full = "".join(r.text for r in p.runs)
            if marcador in full:
                # Conserva formato del primer run como "base"
                base_run = p.runs[0] if p.runs else None
                # Limpia runs
                for r in p.runs:
                    r.text = ""
                # Inserta texto en el primer run (manteniendo su estilo)
                if base_run is None:
                    p.add_run(texto_nuevo)
                else:
                    base_run.text = full.replace(marcador, texto_nuevo)
                return True
    return False

# 4. DETECCIÓN AUTOMÁTICA DE COLUMNA FECHA


def detectar_columna_fecha(df):
    posibles = [c for c in df.columns if 'fecha' in c.lower() and 'rad' in c.lower()]
    if not posibles:
        raise ValueError("❌ No se encontró columna de fecha de radicación en el Excel.")
    return posibles[0]


# 5. FUNCIONES DE TABLAS


def construir_tabla1(df):

    t = (
        df.groupby(['Estado','RAD_ANO'])['Numero radicado']
          .nunique()
          .unstack(fill_value=0)
          .reindex(columns=[2024, 2025], fill_value=0)
    )

    # Total horizontal
    t['Total general'] = t.sum(axis=1)

    # Total vertical
    t.loc['Total general'] = t.sum()

    t = (
        t.reset_index()
         .rename(columns={
             'Estado': 'Estado de la radicación',
             2024: '2024',
             2025: '2025'
         })
    )

    return t

def construir_tabla2(df):

    base = (
        df.groupby(['OFICINA DE GESTION','Tipo','RAD_ANO'])['Numero radicado']
          .nunique()
          .unstack(fill_value=0)
          .reindex(columns=[2024, 2025], fill_value=0)
    )

    base['Total general'] = base.sum(axis=1)
    base = base.reset_index()

    base.columns = [
        'Oficina de Gestión',
        'Tipo de trámite',
        '2024',
        '2025',
        'Total general'
    ]

    base['Nivel'] = base['Oficina de Gestión'].apply(
        lambda x: 'Central' if 'CENTRAL' in str(x).upper() else 'Territorio'
    )

    filas_finales = []

    for nivel in ['Central','Territorio']:

        bloque = base[base['Nivel'] == nivel]

        for _, row in bloque.iterrows():
            filas_finales.append(row.drop('Nivel').to_dict())

        subtotal = {
            'Oficina de Gestión': f'Subtotal {nivel}',
            'Tipo de trámite': '',
            '2024': bloque['2024'].sum(),
            '2025': bloque['2025'].sum(),
            'Total general': bloque['Total general'].sum()
        }

        filas_finales.append(subtotal)

    total_general = {
        'Oficina de Gestión': 'Total general',
        'Tipo de trámite': '',
        '2024': base['2024'].sum(),
        '2025': base['2025'].sum(),
        'Total general': base['Total general'].sum()
    }

    filas_finales.append(total_general)

    return pd.DataFrame(filas_finales)

def construir_tabla3(df):

    df = df.copy()

    df['RES_ANO_GROUP'] = 'En proceso'
    df.loc[df['RES_ANO'] == 2024, 'RES_ANO_GROUP'] = 'Resolución 2024'
    df.loc[df['RES_ANO'] == 2025, 'RES_ANO_GROUP'] = 'Resolución 2025'

    base = (
        df.groupby(['RAD_ANO','OFICINA DE GESTION','Tipo','RES_ANO_GROUP'])
          ['Numero radicado']
          .nunique()
          .unstack(fill_value=0)
          .reindex(columns=['En proceso','Resolución 2024','Resolución 2025'], fill_value=0)
    )

    base['Total general'] = base.sum(axis=1)
    base = base.reset_index()

    base.columns = [
        'Año',
        'Oficina de Gestión',
        'Tipo de trámite',
        'En proceso',
        'Resolución 2024',
        'Resolución 2025',
        'Total general'
    ]

    base['Nivel'] = base['Oficina de Gestión'].apply(
        lambda x: 'Central' if 'CENTRAL' in str(x).upper() else 'Territorio'
    )

    filas_finales = []

    for anio in [2024, 2025]:

        bloque_anio = base[base['Año'] == anio]

        for nivel in ['Central','Territorio']:

            bloque = bloque_anio[bloque_anio['Nivel'] == nivel]

            for _, row in bloque.iterrows():
                filas_finales.append(row.drop('Nivel').to_dict())

            subtotal = {
                'Año': anio,
                'Oficina de Gestión': f'Subtotal {nivel}',
                'Tipo de trámite': '',
                'En proceso': bloque['En proceso'].sum(),
                'Resolución 2024': bloque['Resolución 2024'].sum(),
                'Resolución 2025': bloque['Resolución 2025'].sum(),
                'Total general': bloque['Total general'].sum()
            }

            filas_finales.append(subtotal)

    total_general = {
        'Año': 'Total general',
        'Oficina de Gestión': '',
        'Tipo de trámite': '',
        'En proceso': base['En proceso'].sum(),
        'Resolución 2024': base['Resolución 2024'].sum(),
        'Resolución 2025': base['Resolución 2025'].sum(),
        'Total general': base['Total general'].sum()
    }

    filas_finales.append(total_general)

    return pd.DataFrame(filas_finales)


# 6. FUNCIONES DE GRÁFICAS

def graficar_mes(df, ruta):

    col_fecha = detectar_columna_fecha(df)

    df = df.copy()
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

    # Crear columna numérica del mes
    df['MES_NUM'] = df[col_fecha].dt.month

    # Diccionario correcto (1–12)
    meses_es = {
        1: 'Enero',
        2: 'Febrero',
        3: 'Marzo',
        4: 'Abril',
        5: 'Mayo',
        6: 'Junio',
        7: 'Julio',
        8: 'Agosto',
        9: 'Septiembre',
        10: 'Octubre',
        11: 'Noviembre',
        12: 'Diciembre'
    }

    # Tabla base agrupada
    t = (
        df.groupby(['MES_NUM','RAD_ANO'])['Numero radicado']
          .nunique()
          .unstack(fill_value=0)
          .reindex(columns=[2024, 2025], fill_value=0)
    )

    # Asegurar que estén los 12 meses
    t = t.reindex(range(1,13), fill_value=0)

    # Reemplazar índice numérico por nombre del mes
    t.index = t.index.map(meses_es)

    # Gráfico
    ax = t.plot(kind='bar')

    #plt.title("Radicados por Mes (2024 vs 2025)")
    plt.xlabel("Mes")
    plt.ylabel("Cantidad")
    plt.legend(title="Año")
    plt.xticks(rotation=45)

    # Etiquetas automáticas
    for container in ax.containers:
        ax.bar_label(container, fontsize=8)

    max_val = t.values.max()
    plt.ylim(0, max_val * 1.15 if max_val > 0 else 1)

    plt.tight_layout()
    plt.savefig(ruta)
    plt.close()

def graficar_dia(df, ruta):

    col_fecha = detectar_columna_fecha(df)

    df = df.copy()
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')
    df['DIA_NUM'] = df[col_fecha].dt.weekday

    dias_es = {
        0: 'Lunes',
        1: 'Martes',
        2: 'Miércoles',
        3: 'Jueves',
        4: 'Viernes',
        5: 'Sábado',
        6: 'Domingo'
    }

    df['DIA'] = df['DIA_NUM'].map(dias_es)

    t = (
        df.groupby(['DIA_NUM','DIA','RAD_ANO'])['Numero radicado']
          .nunique()
          .unstack(fill_value=0)
          .reindex(columns=[2024, 2025], fill_value=0)
          .reset_index()
          .sort_values('DIA_NUM')
          .set_index('DIA')
    )

    t = t[[2024, 2025]]

    ax = t.plot(kind='bar')

    #plt.title("Radicados por Día de la Semana (2024 vs 2025)")
    plt.xlabel("Día")
    plt.ylabel("Cantidad")
    plt.legend(title="Año")
    plt.xticks(rotation=45)

    # Etiquetas
    for container in ax.containers:
        ax.bar_label(container, fontsize=8)

    plt.ylim(0, t.values.max() * 1.15 if t.values.max() > 0 else 1)
    plt.tight_layout()
    plt.savefig(ruta)
    plt.close()


# REEMPLAZO REAL DE BLOQUES (PLANTILLA CON TEXTO PLANO)


from docx.shared import Inches
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

def formatear_tabla_word(tabla):

    azul = "2F5496"

    # -------------------------------------------------
    # 0) CENTRAR TODO EL TEXTO DE LA TABLA
    # -------------------------------------------------
    for row in tabla.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # -----------------------------
    # 1) ENCABEZADOS
    # -----------------------------
    for cell in tabla.rows[0].cells:

        tc = cell._tc
        tcPr = tc.get_or_add_tcPr()

        # Limpiar shading previo si existe
        for child in list(tcPr):
            if child.tag == qn('w:shd'):
                tcPr.remove(child)

        shd = OxmlElement('w:shd')
        shd.set(qn('w:fill'), azul)
        tcPr.append(shd)

        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.font.bold = True
                run.font.color.rgb = RGBColor(255, 255, 255)

    # -----------------------------
    # 2) SUBTOTALES Y TOTALES
    # -----------------------------
    for row in tabla.rows[1:]:

        texto_fila = row.cells[0].text.strip().upper()

        if "SUBTOTAL" in texto_fila or "TOTAL GENERAL" in texto_fila:
            for cell in row.cells:

                tc = cell._tc
                tcPr = tc.get_or_add_tcPr()

                # Limpiar shading previo si existe
                for child in list(tcPr):
                    if child.tag == qn('w:shd'):
                        tcPr.remove(child)

                # Fondo blanco
                shd = OxmlElement('w:shd')
                shd.set(qn('w:fill'), "FFFFFF")
                tcPr.append(shd)

                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.font.bold = True
                        run.font.color.rgb = RGBColor(0, 0, 0)

def reemplazar_bloque(doc, titulo_busqueda, tipo, df=None, ruta_imagen=None):

    paragraphs = doc.paragraphs
    inicio = None
    fin = None

    # -------------------------------------------------
    # a. Encontrar párrafo del título
    # -------------------------------------------------
    for i, p in enumerate(paragraphs):
        if titulo_busqueda.strip() in p.text.strip():
            inicio = i
            break

    if inicio is None:
        print(f"⚠ No se encontró el título: {titulo_busqueda}")
        return

    # -------------------------------------------------
    # b. Encontrar siguiente título (Tabla o Imagen)
    # -------------------------------------------------
    for j in range(inicio + 1, len(paragraphs)):
        texto = paragraphs[j].text.strip()
        if texto.startswith("Tabla") or texto.startswith("Imagen"):
            fin = j
            break

    if fin is None:
        fin = len(paragraphs)

    # -------------------------------------------------
    # c. NO eliminar párrafos intermedios
    # Solo determinar posición de inserción
    # -------------------------------------------------

    # Insertaremos justo después del título
    pass

    # -------------------------------------------------
    # d. Insertar nuevo contenido debajo del título
    # -------------------------------------------------

    titulo_parrafo = paragraphs[inicio]

    if tipo == "tabla":

        tabla = doc.add_table(rows=df.shape[0] + 1, cols=df.shape[1])
        tabla.style = "Table Grid"

        # Escribir encabezados
        for col_idx, col in enumerate(df.columns):
            tabla.rows[0].cells[col_idx].text = str(col)

        # Escribir datos
        for row_idx in range(df.shape[0]):
            for col_idx in range(df.shape[1]):
                tabla.rows[row_idx + 1].cells[col_idx].text = str(df.iloc[row_idx, col_idx])

        # Formatear DESPUÉS de escribir
        formatear_tabla_word(tabla)

        titulo_parrafo._element.addnext(tabla._element)

    elif tipo == "imagen":

        nuevo_p = doc.add_paragraph()
        nuevo_p.alignment = WD_ALIGN_PARAGRAPH.CENTER

        run = nuevo_p.add_run()
        run.add_picture(ruta_imagen, width=Inches(4.5))

        titulo_parrafo._element.addnext(nuevo_p._element)

# 7. GENERADOR DE REPORTES


from docx.opc.exceptions import PackageNotFoundError
from docx.shared import Inches
from docx import Document
from zipfile import BadZipFile

def insertar_tabla_word(doc, df, titulo):

    doc.add_heading(titulo, level=2)

    tabla = doc.add_table(rows=df.shape[0] + 1, cols=df.shape[1])
    tabla.style = "Table Grid"

    # Encabezados
    for col_idx, col in enumerate(df.columns):
        tabla.rows[0].cells[col_idx].text = str(col)

    # Datos
    for row_idx in range(df.shape[0]):
        for col_idx in range(df.shape[1]):
            tabla.rows[row_idx + 1].cells[col_idx].text = str(df.iloc[row_idx, col_idx])

import subprocess

def convertir_docx_a_pdf_libreoffice(docx_path, carpeta_pdf):
    """
    Convierte docx a pdf usando LibreOffice (modo headless) y deja el PDF en carpeta_pdf.
    """
    os.makedirs(carpeta_pdf, exist_ok=True)

    cmd = [
        "soffice",
        "--headless",
        "--nologo",
        "--nofirststartwizard",
        "--convert-to", "pdf",
        "--outdir", os.path.abspath(carpeta_pdf),
        os.path.abspath(docx_path)
    ]

    subprocess.run(cmd, check=True)

def obtener_cod_mun(df_mun):
    """
    Retorna COD_MUN del municipio como string limpio.
    Si hay varios valores, toma el primero no nulo.
    """
    if "COD_MUN" not in df_mun.columns:
        return "SIN_COD"
    vals = df_mun["COD_MUN"].dropna().astype(str).unique().tolist()
    if not vals:
        return "SIN_COD"
    # Limpieza: quita .0 si viene como float en Excel
    cod = vals[0]
    if cod.endswith(".0"):
        cod = cod[:-2]
    return cod.strip()

def generar_sistema_reportes(municipio=None):

    df = pd.read_excel(BASE_EXCEL, sheet_name='CRUDOS')
    df = normalizar_columnas(df)
    df = df[df['RAD_ANO'].isin(ANIOS)]

    # ✅ FILTRO: excluir Certificado y Carta en columna Tipo
    df = filtrar_tipos_excluidos(df, columna_tipo="Tipo", excluidos=("Certificado", "Carta"))

    # ✅ Estandarizar Estado
    df = estandarizar_estado(df, col_estado="Estado")

    municipios = [municipio] if municipio else df['NOM_MUN'].unique()

    os.makedirs(CARPETA_RAIZ, exist_ok=True)

    for mun in municipios:

        df_mun = df[df['NOM_MUN'] == mun].copy()
        if df_mun.empty:
            continue

        cod_mun = obtener_cod_mun(df_mun)
        nombre_carpeta = f"{cod_mun}_{mun}"
        base_mun = f"{CARPETA_RAIZ}/{nombre_carpeta}"

        os.makedirs(f"{base_mun}/01_Word", exist_ok=True)
        os.makedirs(f"{base_mun}/02_Graficas", exist_ok=True)
        os.makedirs(f"{base_mun}/03_Pdf", exist_ok=True)

        # ----------------------------
        # Construcción de tablas
        # ----------------------------
        tabla1 = construir_tabla1(df_mun)
        tabla2 = construir_tabla2(df_mun)
        tabla3 = construir_tabla3(df_mun)

        # ----------------------------
        # Generación de gráficas
        # ----------------------------
        ruta_mes = f"{base_mun}/02_Graficas/Radicados_por_mes.png"
        ruta_dia = f"{base_mun}/02_Graficas/Radicados_por_dia.png"

        graficar_mes(df_mun, ruta_mes)
        graficar_dia(df_mun, ruta_dia)

        # ----------------------------
        # Crear o cargar plantilla
        # ----------------------------
        try:
            if not os.path.exists(PLANTILLA_WORD):
                raise FileNotFoundError

            doc = Document(PLANTILLA_WORD)

        except (PackageNotFoundError, FileNotFoundError):
            print("⚠️ Plantilla no encontrada. Se generará documento nuevo.")
            doc = Document()

        # Reemplazar municipio en texto en formato original
        def reemplazar_texto_preservando_formato(doc, texto_busqueda, texto_reemplazo):
            for p in doc.paragraphs:
                for run in p.runs:
                    if texto_busqueda in run.text:
                        run.text = run.text.replace(texto_busqueda, texto_reemplazo)

        # Reemplazar municipio en texto sin perder formato
        reemplazar_texto_preservando_formato(doc, "MPIO", mun.upper())

        # ==================================================
        # INSERTAR ANÁLISIS DESCRIPTIVO (ANTES Y DESPUÉS TABLA 1)
        # ==================================================
        texto_pre, texto_post = construir_textos_analisis(df_mun, mun.upper(), tuple(ANIOS))

        ok1 = reemplazar_marcador_preservando_formato(doc, "[[DESC_RADICADOS]]", texto_pre)
        ok2 = reemplazar_marcador_preservando_formato(doc, "[[ANALISIS_TABLA1]]", texto_post)

        if not ok1:
            print("⚠ No se encontró el marcador [[DESC_RADICADOS]] en la plantilla.")
        if not ok2:
            print("⚠ No se encontró el marcador [[ANALISIS_TABLA1]] en la plantilla.")


        # ORDEN CORRECTO DE INSERCIÓN

        reemplazar_bloque(
            doc,
            "Clasificación de los radicados para trámites de conservación catastral",
            tipo="tabla",
            df=tabla1
        )

        reemplazar_bloque(
            doc,
            "Radicados por tipo y quien resuelve",
            tipo="tabla",
            df=tabla2
        )

        reemplazar_bloque(
            doc,
            "Radicados por mes",
            tipo="imagen",
            ruta_imagen=ruta_mes
        )

        reemplazar_bloque(
            doc,
            "Radicados por día de la semana",
            tipo="imagen",
            ruta_imagen=ruta_dia
        )

        reemplazar_bloque(
            doc,
            "Radicados resueltos desagregados por quien resuelve y por año de resolución",
            tipo="tabla",
            df=tabla3
        )

        # ----------------------------
        # Guardar documento
        # ----------------------------
        word_path = f"{base_mun}/01_Word/{mun}_Informe_Final_ACC_2024_2025.docx"
        doc.save(word_path)

        print(f"✅ Word completo generado para {mun}")

        # ----------------------------
        # Exportar PDF a 03_Pdf
        # ----------------------------
        carpeta_pdf = f"{base_mun}/03_Pdf"
        try:
            convertir_docx_a_pdf_libreoffice(word_path, carpeta_pdf)
            print(f"📄 PDF completo generado para {mun}")
        except Exception as e:
            print(f"⚠ No se pudo generar PDF para {mun}: {e}")



# 8. EJECUCIÓN

# Municipio específico
#generar_sistema_reportes("GUATAVITA")

# Todos los municipios
generar_sistema_reportes()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.8 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.10_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.10) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.10_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.10) ...
Selecting previously unselected package libuno-sal3.
Prepar

In [2]:
# EXPORTAR TODO A ZIP

import shutil
from google.colab import files

zip_base = f"{CARPETA_RAIZ}"                 # carpeta a comprimir
zip_salida = f"{CARPETA_RAIZ}.zip"           # nombre del zip final

# Si ya existe, lo borra para evitar conflictos
if os.path.exists(zip_salida):
    os.remove(zip_salida)

# Crea el zip (sin extensión en make_archive)
shutil.make_archive(zip_base, 'zip', CARPETA_RAIZ)

print(f"✅ ZIP generado: {zip_salida}")

# Descargar al navegador
files.download(zip_salida)

✅ ZIP generado: /content/Reportes_ACC_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>